## Let's talk about **stop words**
You have news articles, and you want to tag or auto tag the company name. how do you do that?
when you are reading the article, you notice certain words such as Elon Musk, gigafactory, model 3. If you encounter those words you know this is Tesla. One way to auto extract entity or tag the topics to a news article is **bag of words**. At this approach, you count the number of words, for example, Tesla appears 14 times, model 3 appears 9 times, and it's obvious a Tesla company.

But if you read the article carefully you will find a lot of other words as well such as wants, time, too, far, over and these words are general English words, which can appear in any article with a equal frequency. So in reality you will have lot of noise.Based on these word count, you can't determine if the article is from Tesla or Apple. So these words are called **stop words**. Wouldn't it be nice if you can filter these words?

![news](/media/elham/Learning/NLP/news_stop_words.jpg)

If you can remove them, you can make your bag of words model little less sparse. So sparse means, in this model you have a lot of noise, the vocabulary is very big, computation time is high, it's a sparse model. Whereas if you remove these words,
your model will get simpler and you will be focusing on things that you care about. So if you have this kind of model, it's a common practice that during pre-processing stage in NLP pipeline, majority of the time they remove stop words.

![pipeline](/media/elham/Learning/NLP/pipeline_2.jpg)

However, there are certain occasion we do not remove stop words : sentiment classification, language translation, Chat bot, Q&A system, any case where value information is lost
![pipeline](/media/elham/Learning/NLP/sentiment_ex.jpg)


## How to remove stop words

In [7]:
import spacy

from spacy.lang.en.stop_words import STOP_WORDS

In [8]:
nlp = spacy.load("en_core_web_sm")
doc = nlp("Pls go ahead with watts. I just wanted to be sure. Do have a great weekend")

for token in doc:
    if token.is_stop:
        print(token)

go
with
I
just
to
be
Do
have
a


**`is_stop` is an attribute of token**

it defines as follow:

`@property`

`def is_stop(self) -> bool: ...`

The @property decorator is a built-in feature that allows you to define methods in a class that can be accessed like attributes.

In [15]:
def preprocess(text):
    doc = nlp(text)
    clean_text = [token.text for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(clean_text)

In [16]:
preprocess("Pls go ahead with watts. I just wanted to be sure. Do have a great weekend")

'Pls ahead watts wanted sure great weekend'

## How do we apply this preprocessing on pandas dataframe?

In [ ]:
# import json
# with open("doj_press.json", "r") as file:
#     data = json.load(file)

In [2]:
# import into pandas

import pandas as pd

df = pd.read_json("doj_press.json", lines=True)
df.head()

,id,title,contents,date,topics,components
0,None,Convicted Bomb Plotter Sentenced to 30 Years,"PORTLAND, Oregon. – Mohamed Osman Mohamud, 23,...",2014-10-01T00:00:00-04:00,[],[National Security Division (NSD)]
1,12-919,$1 Million in Restitution Payments Announced t...,WASHINGTON – North Carolina’s Waccamaw River...,2012-07-25T00:00:00-04:00,[],[Environment and Natural Resources Division]
2,11-1002,$1 Million Settlement Reached for Natural Reso...,BOSTON– A $1-million settlement has been...,2011-08-03T00:00:00-04:00,[],[Environment and Natural Resources Division]
3,10-015,10 Las Vegas Men Indicted \r\nfor Falsifying V...,WASHINGTON—A federal grand jury in Las Vegas...,2010-01-08T00:00:00-05:00,[],[Environment and Natural Resources Division]
4,18-898,$100 Million Settlement Will Speed Cleanup Wor...,"The U.S. Department of Justice, the U.S. Envir...",2018-07-09T00:00:00-04:00,[Environment],[Environment and Natural Resources Division]


In [3]:
df.shape

(13087, 6)

In [4]:
df["topics"].str.len()==0

0         True
1         True
2         True
3         True
4        False
         ...  
13082    False
13083     True
13084    False
13085    False
13086    False
Name: topics, Length: 13087, dtype: bool

In [5]:
df_filtered = df[df["topics"].str.len()!=0] # remove rows that the content of their topics is []
df_filtered.head()

,id,title,contents,date,topics,components
4,18-898,$100 Million Settlement Will Speed Cleanup Wor...,"The U.S. Department of Justice, the U.S. Envir...",2018-07-09T00:00:00-04:00,[Environment],[Environment and Natural Resources Division]
7,14-1412,14 Indicted in Connection with New England Com...,A 131-count criminal indictment was unsealed t...,2014-12-17T00:00:00-05:00,[Consumer Protection],[Civil Division]
19,17-1419,2017 Southeast Regional Animal Cruelty Prosecu...,The United States Attorney’s Office for the Mi...,2017-12-14T00:00:00-05:00,[Environment],"[Environment and Natural Resources Division, U..."
22,15-1562,21st Century Oncology to Pay $19.75 Million to...,"21st Century Oncology LLC, has agreed to pay $...",2015-12-18T00:00:00-05:00,"[False Claims Act, Health Care Fraud]",[Civil Division]
23,17-1404,21st Century Oncology to Pay $26 Million to Se...,21st Century Oncology Inc. and certain of its ...,2017-12-12T00:00:00-05:00,"[Health Care Fraud, False Claims Act]","[Civil Division, USAO - Florida, Middle]"


In [19]:
first_row = df_filtered["contents"].iloc[1]
preprocessed_first_row = preprocess(df_filtered["contents"].iloc[1])
print(f"first row : {first_row}")
print(f"length : {len(first_row)}")
print("-"* 50)
print(f"preprocessed text : {preprocessed_first_row}")
print(f"length : {len(preprocessed_first_row)}")

first row : A 131-count criminal indictment was unsealed today in Boston in connection with the 2012 nationwide fungal meningitis outbreak, the Justice Department announced.  Barry J. Cadden, owner and head pharmacist of New England Compounding Center (NECC) and NECC’s supervisory pharmacist Glenn A. Chin were charged with 25 acts of second-degree murder in Florida, Indiana, Maryland, Michigan, North Carolina, Tennessee and Virginia.    The outbreak was caused by contaminated vials of preservative-free methylprednisolone acetate (MPA) manufactured by NECC, located in Framingham, Massachusetts.  The U.S. Centers for Disease Control and Prevention (CDC) reported that 751 patients in 20 states were diagnosed with a fungal infection after receiving injections of NECC’s MPA.  Of those 751 patients, the CDC reported that 64 patients in nine states died.  Twelve other individuals, all associated with NECC, including six other pharmacists, the director of operations, the national sales directo

In [17]:
df_filtered["key_contents"] = df_filtered["contents"].apply(preprocess)
df_filtered.head(2)


/tmp/ipykernel_17344/1553437691.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["key_contents"] = df_filtered["contents"].apply(preprocess)


,id,title,contents,date,topics,components,key_contents
4,18-898,$100 Million Settlement Will Speed Cleanup Wor...,"The U.S. Department of Justice, the U.S. Envir...",2018-07-09T00:00:00-04:00,[Environment],[Environment and Natural Resources Division],U.S. Department Justice U.S. Environmental Pro...
7,14-1412,14 Indicted in Connection with New England Com...,A 131-count criminal indictment was unsealed t...,2014-12-17T00:00:00-05:00,[Consumer Protection],[Civil Division],131 count criminal indictment unsealed today B...
